<a href="https://colab.research.google.com/github/vunhankhanhmcs3-cmd/master-thesis-mooc-recommendation/blob/main/03_Embedding_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from google.colab import drive

In [ ]:
# 1. Cấu hình
drive.mount('/content/drive')

# Đường dẫn (Cần đảm bảo bạn đã upload file raw vào đây)
BASE_PATH = '/content/drive/MyDrive/01. THESIS/My suggestion/'
PROCESSED_PATH = os.path.join(BASE_PATH, 'data 07.11.2026/processed/')
RELATIONS_PATH = os.path.join(BASE_PATH, 'data 07.11.2026/raw/relations/')
if not os.path.exists(PROCESSED_PATH): os.makedirs(PROCESSED_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 1. Đọc dữ liệu Đồ thị tri thức (Sản phẩm xuất ra từ File 02)
kg_file_path = os.path.join(PROCESSED_PATH, 'kg_final.txt')
print(f"📥 Đang tải đồ thị tri thức từ: {kg_file_path}")

kg_data = pd.read_csv(kg_file_path, sep=' ', header=None)
triplets = torch.tensor(kg_data.values, dtype=torch.long)

# Tính toán số lượng thực thể và mối quan hệ từ file KG sạch (xây từ tập Train)
num_entities = max(kg_data[0].max(), kg_data[2].max()) + 1
num_relations = kg_data[1].max() + 1
EMBEDDING_DIM = 100

print(f"📊 Thông số đồ thị:")
print(f"   - Tổng số thực thể (Entities): {num_entities}")
print(f"   - Tổng số quan hệ (Relations): {num_relations} (Gồm 4 quan hệ xuôi + 4 quan hệ ngược)")
print(f"   - Kích thước không gian nhúng (Embedding Dim): {EMBEDDING_DIM}")

📥 Đang tải đồ thị tri thức từ: /content/drive/MyDrive/01. THESIS/My suggestion/data 07.11.2026/processed/kg_final.txt
📊 Thông số đồ thị:
   - Tổng số thực thể (Entities): 31479
   - Tổng số quan hệ (Relations): 8 (Gồm 4 quan hệ xuôi + 4 quan hệ ngược)
   - Kích thước không gian nhúng (Embedding Dim): 100


In [ ]:
# 2. Định nghĩa kiến trúc thực thể hình học TransE (h + r ≈ t)
class TransE(nn.Module):
    def __init__(self, num_ent, num_rel, dim):
        super().__init__()
        self.entity_emb = nn.Embedding(num_ent, dim)
        self.relation_emb = nn.Embedding(num_rel, dim)

        # Khởi tạo trọng số theo phân phối Xavier Uniform chuẩn học thuật
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self, h, r, t):
        h_vec = self.entity_emb(h)
        r_vec = self.relation_emb(r)
        t_vec = self.entity_emb(t)

        # Tính khoảng cách phi tương đồng dựa trên chuẩn L2-norm
        return torch.norm(h_vec + r_vec - t_vec, p=2, dim=1)

In [ ]:
# 3. Tiến trình Huấn luyện TransE
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🧠 Đang huấn luyện trên thiết bị: {device}")

model = TransE(num_entities, num_relations, EMBEDDING_DIM).to(device)
triplets = triplets.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MarginRankingLoss(margin=1.0) # Margin-based ranking loss

EPOCHS = 20
BATCH_SIZE = 1024

for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(len(triplets))
    total_loss = 0

    for i in tqdm(range(0, len(triplets), BATCH_SIZE), desc=f"Epoch {epoch+1}/{EPOCHS}"):
        batch = triplets[perm[i:i+BATCH_SIZE]]
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]

        # Tạo mẫu tiêu cực ngẫu nhiên (Negative Sampling)
        neg_t = torch.randint(0, num_entities, (len(batch),)).to(device)

        optimizer.zero_grad()
        pos_score = model(h, r, t)
        neg_score = model(h, r, neg_t)

        # Hàm mất mát yêu cầu pos_score nhỏ hơn neg_score ít nhất là một khoảng margin
        target = torch.tensor([-1] * len(batch), dtype=torch.float32).to(device)
        loss = criterion(pos_score, neg_score, target)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"📉 Loss tại Epoch {epoch+1}: {total_loss:.2f}")

🧠 Đang huấn luyện trên thiết bị: cpu


Epoch 1/20: 100%|██████████| 426/426 [00:18<00:00, 22.76it/s]


📉 Loss tại Epoch 1: 203.13


Epoch 2/20: 100%|██████████| 426/426 [00:20<00:00, 21.06it/s]


📉 Loss tại Epoch 2: 92.95


...

📉 Loss tại Epoch 18: 29.69


Epoch 19/20: 100%|██████████| 426/426 [00:21<00:00, 19.73it/s]


📉 Loss tại Epoch 19: 28.33


Epoch 20/20: 100%|██████████| 426/426 [00:20<00:00, 21.23it/s]

📉 Loss tại Epoch 20: 27.07


In [ ]:
# 4. Lưu không gian nhúng phục vụ cho tác nhân Học tăng cường ở File 04
entity_emb_np = model.entity_emb.weight.detach().cpu().numpy()
relation_emb_np = model.relation_emb.weight.detach().cpu().numpy()

np.save(os.path.join(PROCESSED_PATH, 'entity_embeddings.npy'), entity_emb_np)
np.save(os.path.join(PROCESSED_PATH, 'relation_embeddings.npy'), relation_emb_np)

print("💾 Đã lưu thành công entity_embeddings.npy và relation_embeddings.npy vào Google Drive.")
print("🚀 Hệ thống đã sẵn sàng cho bước 04_RL_Training_Eval.ipynb!")

💾 Đã lưu thành công entity_embeddings.npy và relation_embeddings.npy vào Google Drive.
🚀 Hệ thống đã sẵn sàng cho bước 04_RL_Training_Eval.ipynb!
